# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GazalaNK/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Scoring/ranking. I'm giving every page a score for how much it under-captures clicks vs. its position, then ranking pages by that score — not sorting into fixed categories.

In [1]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [6]:
import duckdb
import os
import pandas as pd
from huggingface_hub import notebook_login, get_token

hf_token = get_token()

if hf_token is None:
    print("HF_TOKEN not found. Attempting to log in to Hugging Face.")
    notebook_login()
    hf_token = get_token()

if hf_token is None:
    raise ValueError("HF_TOKEN not set, cannot proceed with data loading.")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

# Correct way to authenticate DuckDB for Hugging Face — use CREATE SECRET
con.sql(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

# Load just March 2026 as your working slice
df_fact = con.sql("""
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

print(df_fact.shape)
print(df_fact.head())

df_fact["position_tier"] = pd.cut(
    df_fact["gsc_avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "20+"]
)
print(df_fact["position_tier"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9841378, 31)
  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True           False                True                <NA>   
1            True           False                True                <NA>   
2            True           False                True                <NA>   
3            True           False                True                <NA>   
4            True           False                True                <NA>   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               20           0              

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target, CTR residual = actual CTR minus the median CTR for that page's position tier. Comes from observed data, not an outside rule. Big negative residual = under-capturing clicks.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

page = df_fact.groupby("content_hash_id").agg(
    impressions=("gsc_impressions","sum"),
    clicks=("gsc_clicks","sum"),
    avg_position=("gsc_avg_position","mean")
).reset_index()
page["ctr"] = page["clicks"] / page["impressions"]
page["position_tier"] = pd.cut(page["avg_position"], bins=[0,3,10,20,1000], labels=["1-3","4-10","11-20","20+"])
page["tier_median_ctr"] = page.groupby("position_tier")["ctr"].transform("median")
page["ctr_residual"] = page["ctr"] - page["tier_median_ctr"]
page.head()


/tmp/ipykernel_1022/3960884829.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  page["tier_median_ctr"] = page.groupby("position_tier")["ctr"].transform("median")


,content_hash_id,impressions,clicks,avg_position,ctr,position_tier,tier_median_ctr,ctr_residual
0,content_000005d4ced12088,86,0,72.854861,0.0,20+,0.0,0.0
1,content_00001e488b74b799,0,0,NaN,NaN,NaN,NaN,NaN
2,content_00007bd2985b77c3,47,0,5.269565,0.0,4-10,0.0,0.0
3,content_00008950670cb6b5,0,0,NaN,NaN,NaN,NaN,NaN
4,content_0000a348850eb1fc,0,0,NaN,NaN,NaN,NaN,NaN


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50 — of the 50 worst-residual pages flagged, how many are real, high-volume candidates worth a reviewer's time.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top50 = page.sort_values("ctr_residual").head(50)
top50[["content_hash_id","impressions","clicks","ctr","ctr_residual"]]


,content_hash_id,impressions,clicks,ctr,ctr_residual
138099,content_6aecb2815b0c4f55,6,0,0.0,0.0
290098,content_e038e67f8fbba4a6,13,0,0.0,0.0
290097,content_e038d9d7fd854f39,2,0,0.0,0.0
290094,content_e03809aac3657962,142,0,0.0,0.0
290090,content_e0368a808eaf4ecd,2,0,0.0,0.0
290088,content_e035a263772f8f53,289,0,0.0,0.0
290086,content_e0351871ee5fc5f3,19,0,0.0,0.0
138024,content_6adbe222995f88c9,2,0,0.0,0.0
138023,content_6adbde5c47264893,162,0,0.0,0.0
138022,content_6adb718b9bed81a5,160,0,0.0,0.0


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page, aggregated: total impressions, total clicks, average position, computed CTR.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(page.shape)
page.head()


(331437, 8)


,content_hash_id,impressions,clicks,avg_position,ctr,position_tier,tier_median_ctr,ctr_residual
0,content_000005d4ced12088,86,0,72.854861,0.0,20+,0.0,0.0
1,content_00001e488b74b799,0,0,NaN,NaN,NaN,NaN,NaN
2,content_00007bd2985b77c3,47,0,5.269565,0.0,4-10,0.0,0.0
3,content_00008950670cb6b5,0,0,NaN,NaN,NaN,NaN,NaN
4,content_0000a348850eb1fc,0,0,NaN,NaN,NaN,NaN,NaN


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A flat CTR cutoff ignores that expected CTR depends on position — low CTR at position #2 means something different than at #15. The median-per-tier below proves CTR varies by position, so one fixed threshold can't work everywhere.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

page.groupby("position_tier")["ctr"].median()

/tmp/ipykernel_1022/2627529427.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  page.groupby("position_tier")["ctr"].median()


,ctr
position_tier,
1-3,0.0
4-10,0.0
11-20,0.0
20+,0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.